# Neutral-Atom MHT Simulation

This notebook runs the simulation-only workflow: generate frame truth, add blur and grain, then track the simulated images.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd()
if not (project_root / "pyproject.toml").is_file():
    raise RuntimeError("Open user_notebook.ipynb from the repository root.")

src_path = str(project_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from neutral_atom_mht import ClassicalSolver, HPC, HPCConfig, SyntheticDataConfig, SyntheticDataGenerator

In [ ]:
config = SyntheticDataConfig(
    frame_count=6,
    object_count=4,
    image_shape=(128, 160),
    gaussian_bluriness=1.0,
    grainyness=2.0,
    speed_px_per_frame=2.5,
    seed=4,
)

frames = list(SyntheticDataGenerator(config).iter_simulated_frames())
simulation_summary = {
    "frames": len(frames),
    "objects": config.object_count,
    "image_shape": frames[0].image.shape,
    "truth_ids_frame_0": sorted(set(frames[0].labels.ravel()) - {0}),
    "gaussian_bluriness": config.gaussian_bluriness,
    "grainyness": config.grainyness,
}
simulation_summary

In [ ]:
tracker = HPC(HPCConfig(), sequence=config.sequence)
solver = ClassicalSolver(maximum_component_nodes=12)
result = tracker.run_sequence((frame.image for frame in frames), solver, start_frame=0)

tracking_summary = {
    "processed_frames": [step.frame for step in result.steps],
    "final_track_count": len(result.final_tracks),
    "solver": result.solver_name,
    "last_frame": tracker.last_frame,
    "assigned_observations": [step.assigned_observation_ids for step in result.steps],
}
tracking_summary

In [ ]:
prepared = HPC(HPCConfig(), sequence=config.sequence).prepare_frame(frames[0].image, frame=0)
{
    "detections": len(prepared.observations),
    "graph_nodes": len(prepared.graph.nodes),
    "graph_edges": len(prepared.graph.edges),
    "source_state_fingerprint": prepared.source_state_fingerprint,
}